# Member 2 — Explainable Ayurvedic Relation Extraction

| | |
|---|---|
| **Model** | `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext` (renamed upstream to *BiomedBERT*; old id still resolves with transformers ≥ 4.22) |
| **Corpus** | Charak Samhita New Edition — English translation of **Nidana Sthana** and **Chikitsa Sthana**, verse-referenced (carakasamhitaonline.com, CC BY-NC-SA 4.0). The originally proposed `shinigamiRaj/IndianVedasOriginal` is checked in §3 and used only if it passes verification. |
| **Task** | Entity-marked relation classification over gazetteer-detected entity pairs |
| **Labels** | `TREATS`, `REDUCES`, `AGGRAVATES`, `HAS_PROPERTY`, `NO_RELATION` |
| **Explainability** | Attention rollout from `[CLS]` + exact evidence sentence with character offsets |
| **Exports** | `models/model2.py`, `weights/model2/` |

This notebook is the experimental front-end. All reusable logic lives in the `training/` package and in `models/model2.py`, so the notebook and the production module cannot drift apart.

> **Scientific caveat.** Training labels are produced by rules (*weak labels*). Scores against weak labels measure agreement with the rules, not real-world accuracy. Report gold-label results only after manual review (§6). Nothing here is medical advice or clinical validation.

## 0. Audit of the previous notebook (summary)

Fixed in this version (details in `README.md`):

1. **Attention rollout silently returned no cues.** Recent `transformers` use SDPA attention, which returns no attention weights; `output_attentions=True` yields an empty tuple. The model is now loaded with `attn_implementation="eager"` and the module raises if attentions are missing.
2. **Untrained fallback.** The exported module fell back to the *base* checkpoint when `weights/model2/` was missing, producing predictions from a randomly initialised classification head. It now raises `Model2NotAvailableError` instead.
3. **Model loaded and tokenizer resized at import time**; now lazy, thread-safe, no work on import.
4. **Corpus filter trusted keyword matches** in text bodies; replaced by a verified, verse-referenced source with metadata-based checks.
5. **Random row split** leaked near-duplicate passages; splits are now by chapter.
6. **No negatives** were produced (`NO_RELATION` never appeared in training), so the classifier could not learn to reject pairs. Weak negatives are now built only from sentences with no relational language; ambiguous pairs go to review instead.
7. **Evidence was the whole input**; it is now the exact supporting sentence/clause with offsets.
8. **Cues were raw WordPiece tokens** (e.g. `##ra`) not aligned to the input; they are now whole words that are exact substrings of the input.
9. Relation-type checks used case-sensitive set membership (`"kapha" not in {"Kapha"}`), dropping valid predictions; replaced by typed entities and label constraints.

## 1. Setup

In [ ]:
# Run from the repository root (member2-delivery/). In Colab, first clone/upload the repo and `%cd` into it.
# %pip install -q -r requirements.txt
import json, logging, os, sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "models" / "model2.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

import torch, transformers
from training.train import TrainConfig, set_seed
CFG = TrainConfig()          # all hyper-parameters in one place
set_seed(CFG.seed)
print("root:", ROOT, "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda:", torch.cuda.is_available())

## 2. Model verification

Checks that the checkpoint downloads, what it is, and how its English biomedical WordPiece vocabulary handles Ayurvedic (romanised Sanskrit) terms. PubMedBERT is an **English** model: it cannot read Devanagari or IAST verse text meaningfully, which is why only English translations are used. Its classification head is untrained, so fine-tuning is required.

In [ ]:
from transformers import AutoConfig, AutoTokenizer
cfg = AutoConfig.from_pretrained(CFG.base_checkpoint)
tok = AutoTokenizer.from_pretrained(CFG.base_checkpoint)
print({k: getattr(cfg, k) for k in ["model_type", "vocab_size", "num_hidden_layers", "num_attention_heads", "hidden_size", "max_position_embeddings"]})
for term in ["haridra", "kushtha", "kapha", "guduchi", "tridosha", "ushna virya", "fever", "diarrhea", "कुष्ठ"]:
    print(f"{term:12s} -> {tok.tokenize(term)}")
# Expect Sanskrit terms to fragment into several word pieces and Devanagari to map to [UNK]-like pieces.

## 3. Verify the originally proposed corpus

`shinigamiRaj/IndianVedasOriginal` is checked for (a) records whose **metadata** identifies Charaka Samhita (keyword hits in the body do not count), (b) English content, (c) chapter/verse references, (d) a declared licence. It is used only if `suitable` is `True`.

In [ ]:
from training.inspect_hf_candidate import inspect
try:
    hf_report = inspect("shinigamiRaj/IndianVedasOriginal")
    print(json.dumps({k: hf_report[k] for k in ["license", "suitable", "reasons"]}, indent=2))
    for name, s in hf_report["splits"].items():
        print(name, {k: s[k] for k in ["rows", "metadata_columns", "charaka_by_metadata", "charaka_english_records"]})
except Exception as exc:  # record the failure; do not fall back silently
    hf_report = {"suitable": False, "reasons": [f"inspection failed: {exc!r}"]}
    print(hf_report)

## 4. Load the verified Charaka corpus (Nidana & Chikitsa Sthana)

Pages are discovered from the sthana index pages, verified by the in-page header *"<Sthana> Sthana Chapter n"*, cached under `data/raw/` (git-ignored), and parsed into English verse units with chapter, verse reference, URL, revision id, DOI and licence.

In [ ]:
from training.corpus_charaka import load_corpus, save_units, read_units
UNITS = Path("data/processed/charaka_units.jsonl")
if not UNITS.exists():
    units = load_corpus(["Nidana", "Chikitsa"], cache_dir="data/raw/charaka")
    manifest = save_units(units, UNITS)
units = read_units(UNITS)
manifest = json.loads(UNITS.with_suffix(".manifest.json").read_text())
print("verse units:", len(units), "| chapters:", len(manifest["chapters"]), "| sha256:", manifest["sha256"][:16])
import collections
print(collections.Counter(u.sthana for u in units))
print(units[0].source_ref)
print(units[0].text[:300])

In [ ]:
# Language check on the extracted units (should be English, Latin script).
from training.inspect_hf_candidate import _script_profile
profiles = [_script_profile(u.text) for u in units]
print("mean Devanagari share:", sum(p["devanagari"] for p in profiles) / len(profiles))
print("mean English stop-word rate:", sum(p["stopword_rate"] for p in profiles) / len(profiles))

## 5. Entities, candidate pairs and weak labels

* **NER** is a typed gazetteer (`models/model2.py::ENTITY_LEXICON`), not a learned model. Ambiguous strings (e.g. *kushtha* = skin disease **or** the herb *Saussurea*) keep both types.
* **Candidates**: same-sentence pairs whose types fit a relation signature (`RELATION_SIGNATURES`).
* **Weak labels** (`training/weak_labels.py`): `weak_positive` (directional trigger bound between the entities in one clause), `weak_negative` (no relational or causal language anywhere in the sentence), `uncertain` (everything else; excluded from training, sent to review).
* Causal statements (*X causes kushtha*) dominate Nidana Sthana but have no label in the team schema; they are flagged `causal_out_of_schema`.

In [ ]:
from training.weak_labels import build_candidates, to_records
from training.data import export_review_sheet, write_jsonl
import pandas as pd

records = to_records(build_candidates(units))
write_jsonl(records, "data/processed/candidates.jsonl")
n = export_review_sheet(records, "data/annotation/review_sheet.csv")
df = pd.DataFrame(records)
print("candidates:", len(df), "| review sheet rows:", n)
display(pd.crosstab(df["label_origin"], df["weak_relation"].fillna("—")))
display(df["rule"].value_counts().head(15))
display(df[df.label_origin == "weak_positive"][["source_ref", "head", "weak_relation", "tail", "sentence"]].head(10))

## 6. Manual review (gold labels)

1. Open `data/annotation/review_sheet.csv`. Fill `gold_relation` (one of the five labels), `reviewer`, `review_note`.
2. Two reviewers label the same ≥ 200 rows independently, preferably from **whole chapters** reserved for testing; compute agreement with `training.data.cohens_kappa` and adjudicate.
3. Save the adjudicated sheet as `data/annotation/reviewed.csv` and rerun this cell. Gold labels override weak labels; the gold test set is written to `data/annotation/gold_test.jsonl`.

In [ ]:
from training.data import merge_gold, read_review_sheet, cohens_kappa
REVIEWED = Path("data/annotation/reviewed.csv")
GOLD_TEST = None
if REVIEWED.exists():
    records = merge_gold(records, read_review_sheet(REVIEWED))
    write_jsonl(records, "data/processed/candidates.jsonl")
    gold = [r for r in records if r.get("gold_relation")]
    write_jsonl(gold, "data/annotation/gold_test.jsonl")
    GOLD_TEST = "data/annotation/gold_test.jsonl"
    print("gold-labelled candidates:", len(gold))
else:
    print("No reviewed file yet: evaluation below is against WEAK labels only.")

## 7. Fine-tune

Chapter-level train/dev/test split (no chapter in two splits), class-weighted cross-entropy, linear warm-up/decay, early stopping on dev macro-F1, seed fixed in `TrainConfig`. A GPU is recommended (≈110M parameters). Training refuses to run on fewer than `min_examples` trainable examples.

In [ ]:
from training.train import train
meta = train("data/processed/candidates.jsonl", "weights/model2", CFG, gold_test=GOLD_TEST)
print(json.dumps(meta["split_report"]["labels_missing_from_test"]))
print(json.dumps(meta["history"], indent=2))

## 8. Evaluation and error analysis

`weak_label_test` = agreement with the rules on held-out chapters (**not verified accuracy**). `gold_test` (if present) is the number to report, together with the rule baseline.

In [ ]:
ev = meta["evaluation"]
print("WEAK-label test:", {k: ev["weak_label_test"][k] for k in ["n", "accuracy", "macro_f1", "macro_f1_positive_labels"]})
display(pd.DataFrame(ev["weak_label_test"]["per_label"]).T)
display(pd.DataFrame(ev["weak_label_test"]["confusion"]).T)
if ev["gold_test"]:
    from training.evaluate import rule_baseline
    from training.data import read_jsonl
    print("GOLD test:", {k: ev["gold_test"][k] for k in ["n", "accuracy", "macro_f1"]})
    print("Rule baseline on GOLD:", rule_baseline(read_jsonl(GOLD_TEST))["macro_f1"])
errs = Path("weights/model2/test_errors_weak.csv")
if errs.exists():
    display(pd.read_csv(errs).head(20))

## 9. Exported module, explainability and contract check

`models.model2.predict` loads `weights/model2/` lazily (never retrains, never substitutes the base model). `predict_detailed` adds offsets, entity types, all class probabilities and per-word attention-rollout scores.

**Reading the cues.** Rollout estimates how much each input word contributes to the `[CLS]` representation through the attention graph. It is class-agnostic (identical for every label of one input), ignores value vectors and feed-forward layers, and is not a causal explanation. Use it as a highlighting aid alongside the evidence sentence.

In [ ]:
import html
import models.model2 as m2
m2.reset()
sample = "Haridra is useful in Kushta and reduces Kapha."
print(json.dumps(m2.predict(sample), indent=2))

def highlight(d):
    text, spans = d["input_text"], d["token_attributions"]
    top = max((w["score"] for w in spans), default=1) or 1
    out, pos = [], 0
    for w in spans:
        out.append(html.escape(text[pos:w["start"]]))
        alpha = w["score"] / top
        out.append(f'<span style="background: rgba(255,140,0,{alpha:.2f})" title="{w["score"]}">{html.escape(w["token"])}</span>')
        pos = w["end"]
    out.append(html.escape(text[pos:]))
    return f'<p><b>{d["head"]} —{d["relation"]}→ {d["tail"]}</b> ({d["confidence"]})<br>' + "".join(out) + "</p>"

from IPython.display import HTML
HTML("".join(highlight(d) for d in m2.predict_detailed(sample)))

In [ ]:
# Interface tests (synthetic fixture; no training needed). Add MODEL2_TEST_REAL_WEIGHTS=1 to also test weights/model2/.
import subprocess
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/"], capture_output=True, text=True).stdout[-2000:])

## 10. Hand-over checklist

- [ ] `weights/model2/member2_metadata.json` has `"status": "trained"` and records data hash, split report and evaluation label type.
- [ ] `MODEL2_TEST_REAL_WEIGHTS=1 pytest -q tests/` passes.
- [ ] Gold-label evaluation completed and reported with the rule baseline; weak-label scores labelled as such.
- [ ] `confidence` described as uncalibrated softmax probability.
- [ ] Corpus attribution (CC BY-NC-SA 4.0) kept in README and any published artifacts.